In [1]:
import os
from pathlib import Path

import dotenv
import numpy as np
import pandas as pd

# Load environment variables
dotenv.load_dotenv()

# Set up output folder
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])
FILE_DIR = Path(OUTPUT_FOLDER / "individual_runs")

In [2]:
MANIFEST_DIR = FILE_DIR / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# magicc_flag is always the final "_<FLAG>" suffix on the filename
# ({model}_{scenario}_{flag}.csv - see 301_run_magicc.ipynb); any NZCO2/NZKyoto
# variant marker is already part of the scenario name itself (read from the file's
# own "scenario" column below), not a separate suffix token.
MAGICC_FLAGS = ("ANTHROPOGENIC", "CO2", "GHG")

def extract_flag(stem):
    for flag in MAGICC_FLAGS:
        if stem.endswith(f"_{flag}"):
            return flag
    return None

records = []
files = sorted(FILE_DIR.glob("*.csv"))
print(f"Found {len(files)} CSV files")

for f in files:
    magicc_flag = extract_flag(f.stem)
    if magicc_flag is None:
        print(f"Skipping unrecognised: {f.name}")
        continue
        
    try:
        data = pd.read_csv(f, usecols=["model", "scenario", "run_id"])
        model = data["model"].iloc[0]
        scenario = data["scenario"].iloc[0]
        ensemble_members = data["run_id"].nunique()
    except Exception as e:
        print(f"Could not read {f.name}: {e}")
        continue

    records.append({
        "model": model,
        "scenario": scenario,
        "ensemble_members": ensemble_members,
        "n_processes": 10,
        "magicc_flag": magicc_flag,
        "output_filename": f.name,
    })

df = pd.DataFrame(records)

n_files = len(df)
n_scenarios = df[["model", "scenario"]].drop_duplicates().shape[0]
print(f"\nParsed {n_files} files across {n_scenarios} unique model-scenario combinations")
print(f"\nFlag counts:\n{df['magicc_flag'].value_counts().to_string()}")

# Check for duplicates (same model/scenario/flag appearing twice)
dupes = df.duplicated(subset=["model", "scenario", "magicc_flag"], keep=False)
if dupes.any():
    print(f"\nWARNING: {dupes.sum()} duplicate model/scenario/flag combinations:")
    print(df[dupes].to_string())

manifest_path = MANIFEST_DIR / "regenerated_manifest.csv"
df.to_csv(manifest_path, index=False)
print(f"\nManifest saved to: {manifest_path}")

Found 27 CSV files

Parsed 27 files across 9 unique model-scenario combinations

Flag counts:
magicc_flag
ANTHROPOGENIC    9
CO2              9
GHG              9

Manifest saved to: /Users/hoegner/GitHub/ar7_netzero_assessment/analysis/output_revision/individual_runs/manifests/regenerated_manifest.csv


In [4]:
# Check which (model, scenario) rows are missing one or more of the three MAGICC_FLAGS
# 301_run_magicc.ipynb always tries to produce all three per scenario in one execution
# - a partial failure inside that notebook would still leave the other flags' files
# behind, so this can't be detected purely from "is the file missing".

flags_by_scenario = df.groupby(["model", "scenario"])["magicc_flag"].apply(set)
incomplete = flags_by_scenario[flags_by_scenario.apply(lambda s: s != set(MAGICC_FLAGS))]
print(f"{len(incomplete)} (model, scenario) rows missing one or more flags:")
incomplete.apply(lambda s: sorted(set(MAGICC_FLAGS) - s))

0 (model, scenario) rows missing one or more flags:


Series([], Name: magicc_flag, dtype: object)